# `debug.ipynb` — revisao visual, etapa por etapa, do pipeline de segmentacao

**Objetivo do notebook:** dar ao avaliador uma visao honesta de *cada etapa* do `pdiseg`,
renderizando os mesmos primitivos PDI que o pipeline avaliado executa (ADR 0005) — nao uma
reimplementacao paralela que poderia divergir.

**Como julgar (sem ground truth, ADR 0004):** nao existe gabarito por etiqueta na base, entao
cada etapa e avaliada por *fazer o seu trabalho*, sempre sobre um **par de frames facil + dificil**
(caixa selada vs. saco amassado) — os dois regimes que a ADR 0004 identificou.

**Stage 1 em transicao (ADR 0006):** a heuristica antiga (densidade de texto) fundia a caixa
inteira num unico blob nos frames com brilho de plastico; a etiqueta de nome e texto claro sobre
um **badge escuro**, entao a *escuridao* a separa do brilho (claro-sobre-claro). Este notebook roda
o **pipeline escuro proposto** (`detect_dark_badges`) de ponta a ponta para voce calibra-lo; o
`detect_name_labels` avaliado ainda usa o caminho antigo ate voce aprovar a promocao.

Rode de cima para baixo. Ajuste `STAGE1_PERCENTILE` (celula do amostrador) e re-execute.

## Configuracao — imports, caminhos e funcoes auxiliares

Define o dataset, os atalhos para os dois regimes (facil/dificil, ADR 0004) e os auxiliares de
plotagem (`show`, `draw_boxes`) reusados por todas as celulas abaixo.

In [ ]:
%matplotlib inline
from pathlib import Path

import imageio.v3 as iio
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Rectangle

import pdiseg

DATASET = Path('data/Train_and_Validation')

EASY_HINT = 'Selado'                     # caixa selada — regime facil (ADR 0004)
HARD_HINT = '93000088_Peito_Congelado'   # saco amassado — regime dificil (ADR 0004)
_SUFFIXES = {'.jpg', '.jpeg', '.png'}


def classes():
    return sorted(p.name for p in DATASET.iterdir() if p.is_dir())


def frames(c):
    return sorted(p for p in (DATASET / c).iterdir()
                  if p.is_file() and p.suffix.lower() in _SUFFIXES)


def find_class(hint):
    for name in classes():
        if hint.lower() in name.lower():
            return name
    raise LookupError(f'nenhuma classe casa com {hint!r}; disponiveis: {classes()}')


def load_gray(path):
    img = iio.imread(path)
    if img.ndim > 2:
        img = img[..., 0]
    return img.astype(np.uint8)


def show(ax, img, title, cmap='gray'):
    ax.imshow(img, cmap=cmap, vmin=0, vmax=255 if img.dtype == np.uint8 else None)
    ax.set_title(title, fontsize=10)
    ax.axis('off')


def draw_boxes(ax, boxes, color, labels=None, lw=1.8, ls='-'):
    for i, (x, y, w, h) in enumerate(boxes):
        ax.add_patch(Rectangle((x, y), w, h, fill=False, edgecolor=color, linewidth=lw, linestyle=ls))
        if labels is not None:
            ax.text(x, max(y - 4, 0), labels[i], color=color, fontsize=7, backgroundcolor='black')


print('classes disponiveis:')
for c in classes():
    print('  ', c)

## Amostrador dos dois regimes (+ o knob da Stage 1)

**Objetivo:** fixar um par facil + dificil que todas as celulas abaixo reutilizam, e expor o unico
parametro de calibracao da Stage 1.

`STAGE1_PERCENTILE` e o limiar do badge escuro (ADR 0006): a `dark_mask` marca os *X% mais escuros*
do frame. Menor = so o mais escuro = menos candidatos; maior = mais candidatos (e mais fusao).
`stage1(working)` e exatamente o que toda celula abaixo chama.

In [ ]:
EASY_CLASS = find_class(EASY_HINT)
HARD_CLASS = find_class(HARD_HINT)
EASY_IDX = 0
HARD_IDX = 0

STAGE1_PERCENTILE = 20   # <-- ajuste aqui (knob da Stage 1, ADR 0006)


def stage1(working):
    'Stage 1 proposta: candidatos de badge escuro — a rede de recall em avaliacao (ADR 0006).'
    return pdiseg.detect_dark_badges(working, percentile=STAGE1_PERCENTILE)


EASY_PATH = frames(EASY_CLASS)[EASY_IDX]
HARD_PATH = frames(HARD_CLASS)[HARD_IDX]
PAIR = [
    ('EASY  ' + EASY_CLASS, EASY_PATH, load_gray(EASY_PATH)),
    ('HARD  ' + HARD_CLASS, HARD_PATH, load_gray(HARD_PATH)),
]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, (label, path, frame) in zip(axes, PAIR):
    show(ax, frame, f'{label}\n{path.name}')
plt.tight_layout(); plt.show()

## Etapa 0 — `preprocess`  (cru -> mediana -> equalizacao + mascara do FPS)

**Objetivo:** limpar ruido e neutralizar o overlay de FPS antes de detectar.
**Heuristica:** filtro de mediana 3x3 (remove sal-e-pimenta sem borrar bordas) + `equalize_hist`
(espalha o histograma) + preenche a regiao do FPS com a mediana do frame (ADR 0004: borda dura de
zero virava cluster fantasma no canto superior esquerdo).

Nota (ADR 0006): o limiar do badge escuro e um *percentil*, **invariante** a equalizacao monotonica
— entao a equalizacao nem ajuda nem atrapalha a Stage 1; ela permanece para o split Otsu da Etapa 2.
**PASSA:** caixa do FPS achatada, sem estrutura destruida.

In [ ]:
from scipy.ndimage import median_filter
from skimage.exposure import equalize_hist
from skimage.util import img_as_ubyte

fig, axes = plt.subplots(len(PAIR), 3, figsize=(14, 4.2 * len(PAIR)))
for row, (label, path, frame) in zip(axes, PAIR):
    median_only = median_filter(frame, size=3)
    final = pdiseg.preprocess(frame)
    show(row[0], frame, f'{label}\ncru')
    show(row[1], median_only, 'so mediana')
    show(row[2], final, 'preprocess (mediana+equaliza+FPS)')
plt.tight_layout(); plt.show()

## Etapa 1 — rede de recall por badge escuro  (`detect_dark_badges`, ADR 0006)

**Objetivo:** localizar **todo** candidato a etiqueta de nome; superdeteccao e esperada e e a Etapa 3
que rejeita (a Stage 1 e uma *rede de recall*, ADR 0001).
**Heuristica (ADR 0006):** a etiqueta e texto claro sobre um **badge escuro**, enquanto o brilho do
plastico e claro-sobre-claro — entao a escuridao discrimina. Primitivos (ADR 0005):
`dark_mask` (X% mais escuros) -> `open_mask` (remove respingos de brilho) -> `close_mask(9x9)`
(solidifica o badge) -> `label_components` -> `boxes_from_components`.

**PASSA:** toda etiqueta de nome *real* fica dentro de >=1 caixa amarela. **FALHA:** uma etiqueta
real e **perdida**. As caixas **vermelhas tracejadas** sao os candidatos ANTIGOS por densidade de
texto (`detect_clusters`) — mostradas so para ver por que pivotamos (engolem o frame).

In [ ]:
fig, axes = plt.subplots(len(PAIR), 5, figsize=(20, 4.2 * len(PAIR)))
for row, (label, path, frame) in zip(axes, PAIR):
    working = pdiseg.preprocess(frame)
    dm = pdiseg.dark_mask(working, percentile=STAGE1_PERCENTILE)
    opened = pdiseg.open_mask(dm)
    closed = pdiseg.close_mask(opened, structure=np.ones((9, 9)))
    comps = pdiseg.label_components(closed)
    candidates = pdiseg.boxes_from_components(comps)
    assert candidates == stage1(working)  # identico a Stage 1 proposta (mesmo codigo, ADR 0005)
    old = pdiseg.detect_clusters(working)  # densidade de texto, so p/ contraste (ADR 0006)
    show(row[0], dm, f'{label}\n1a dark_mask p{STAGE1_PERCENTILE}')
    show(row[1], opened, '1b open_mask')
    show(row[2], closed, '1c close_mask 9x9')
    show(row[3], np.where(comps > 0, comps, np.nan), '1d componentes', cmap='nipy_spectral')
    show(row[4], working, f'1e candidatos  escuro={len(candidates)}  (texto antigo={len(old)})')
    draw_boxes(row[4], old, 'red', lw=1.2, ls='--')
    draw_boxes(row[4], candidates, 'yellow')
plt.tight_layout(); plt.show()
print('FALHA so se uma etiqueta de nome real NAO tiver caixa amarela. '
      'Vermelho tracejado = texto antigo (por que pivotamos).')

### Stage 1 interativo - escolha a fonte, navegue frame a frame, gire o percentil

- **fonte**: `1 demo por classe` (1 imagem de cada classe, p/ comparar classes) · `classe inteira` (todos os frames de uma classe) · `full (todas)`.
- O slider **frame** se ajusta sozinho ao tamanho da fonte; use **◀ / ▶** para passo a passo. `classe` so fica ativo no modo `classe inteira`.
- **percentil** ajusta o `dark_mask` ao vivo (ADR 0006). Painel grande: `cand` (amarelo), `kept` (verde, sobrevive a Stage 3) e, opcional, o texto antigo (vermelho - desligado por padrao, e lento).
- Diagnostico no titulo: `% escuro` (quanto a mascara dispara) e `maior` (fracao do frame coberta pelo maior candidato). `maior > 40%` marca **MERGE** - sinal de que o percentil global nao separa o badge naquela classe.

> Os controles usam `observe`/`continuous_update=False`, entao so recalcula quando voce solta o slider - bem mais leve para varrer muitas classes/imagens.


In [ ]:
import ipywidgets as widgets
from IPython.display import display


def _source_set(mode, cls):
    if mode == '1 demo por classe':
        return [(c, frames(c)[0]) for c in classes() if frames(c)]
    if mode == 'classe inteira':
        return [(cls, p) for p in frames(cls)]
    return [(c, p) for c in classes() for p in frames(c)]


# --- controles ---------------------------------------------------------------
w_fonte = widgets.Dropdown(options=['1 demo por classe', 'classe inteira', 'full (todas)'],
                           value='1 demo por classe', description='fonte')
w_classe = widgets.Dropdown(options=classes(), value=HARD_CLASS, description='classe')
w_perc = widgets.IntSlider(min=5, max=40, step=1, value=STAGE1_PERCENTILE,
                           description='percentil', continuous_update=False)
w_frame = widgets.IntSlider(min=0, max=0, step=1, value=0, description='frame',
                            continuous_update=False, layout=widgets.Layout(width='420px'))
w_prev = widgets.Button(description='◀', layout=widgets.Layout(width='42px'))
w_next = widgets.Button(description='▶', layout=widgets.Layout(width='42px'))
w_oldtext = widgets.Checkbox(value=False, description='texto antigo (lento)')
out = widgets.Output()


def _items():
    return _source_set(w_fonte.value, w_classe.value)


def _render(*_):
    items = _items()
    with out:
        out.clear_output(wait=True)
        if not items:
            print('conjunto vazio'); return
        i = min(w_frame.value, len(items) - 1)
        cls_name, path = items[i]
        working = pdiseg.preprocess(load_gray(path))
        p = w_perc.value
        dm = pdiseg.dark_mask(working, percentile=p)
        opened = pdiseg.open_mask(dm)
        closed = pdiseg.close_mask(opened, structure=np.ones((9, 9)))
        comps = pdiseg.label_components(closed)
        cand = pdiseg.boxes_from_components(comps)
        kept = pdiseg.keep_label_clusters(cand)
        frac = max(((bw * bh) / working.size for *_, bw, bh in cand), default=0.0)
        # 4 mascaras pequenas em cima, overlay grande embaixo (mais facil de ler)
        fig = plt.figure(figsize=(18, 8))
        gs = fig.add_gridspec(2, 4, height_ratios=[1, 2.2])
        a = [fig.add_subplot(gs[0, k]) for k in range(4)]
        show(a[0], dm, f'1a dark_mask p{p}\n{100 * dm.mean():.0f}% escuro')
        show(a[1], opened, '1b open')
        show(a[2], closed, '1c close 9x9')
        show(a[3], np.where(comps > 0, comps, np.nan), '1d components', cmap='nipy_spectral')
        big = fig.add_subplot(gs[1, :])
        show(big, working, f'{cls_name}   [{i + 1}/{len(items)}]   '
                           f'cand={len(cand)}  kept={len(kept)}  maior={frac:.0%}')
        if w_oldtext.value:
            draw_boxes(big, pdiseg.detect_clusters(working), 'red', lw=1.0, ls='--')
        draw_boxes(big, cand, 'yellow')
        draw_boxes(big, kept, 'lime', lw=2.4)
        plt.tight_layout(); plt.show()
        flag = 'vazio' if not cand else ('MERGE' if frac > 0.40 else 'ok')
        print(f'{path.name}    [{flag}]    fonte={w_fonte.value}')


def _resync(*_):
    n = len(_items())
    w_frame.max = max(n - 1, 0)
    if w_frame.value > w_frame.max:
        w_frame.value = w_frame.max
    w_classe.disabled = (w_fonte.value != 'classe inteira')
    _render()


def _step(delta):
    return lambda _: setattr(w_frame, 'value',
                             min(max(w_frame.value + delta, 0), w_frame.max))


w_fonte.observe(_resync, 'value')
w_classe.observe(_resync, 'value')
w_perc.observe(_render, 'value')
w_frame.observe(_render, 'value')
w_prev.on_click(_step(-1))
w_next.on_click(_step(1))

display(widgets.VBox([
    widgets.HBox([w_fonte, w_classe]),
    widgets.HBox([w_perc, w_prev, w_frame, w_next]),
    w_oldtext,
]), out)
_resync()


## Etapa 1 — varredura de percentil (calibrar `STAGE1_PERCENTILE`)

**Objetivo:** escolher o limiar do badge escuro. Mesmos frames, candidatos em varios percentis.
**Heuristica (ADR 0004, recall-favoring):** pegue o **menor** percentil que ainda poe uma caixa em
toda etiqueta real; note a superdeteccao crescer conforme ele sobe. Amarelo = todos os candidatos
escuros; verde = sobrevivem a geometria da Etapa 3.

In [ ]:
SWEEP = [15, 20, 25, 30]
fig, axes = plt.subplots(len(PAIR), len(SWEEP), figsize=(5 * len(SWEEP), 4.2 * len(PAIR)))
for row, (label, path, frame) in zip(axes, PAIR):
    working = pdiseg.preprocess(frame)
    for ax, pct in zip(row, SWEEP):
        cands = pdiseg.detect_dark_badges(working, percentile=pct)
        kept = pdiseg.keep_label_clusters(cands)
        show(ax, working, f'{label[:14]}  p{pct}\ncand={len(cands)} kept={len(kept)}')
        draw_boxes(ax, cands, 'yellow', lw=1.0)
        draw_boxes(ax, kept, 'lime', lw=2.0)
plt.tight_layout(); plt.show()
print('amarelo = todos os candidatos escuros, verde = sobrevivem a geometria da Etapa 3. '
      'Trave o percentil na celula do amostrador.')

## Etapa 3 — `keep_label_clusters`  (rejeicao geometrica frouxa)

**Objetivo:** descartar so os nao-rotulos inequivocos (fusoes de frame inteiro, codigos de barras,
bordas finas) preservando recall (ADR 0004).
**Heuristica:** mantem caixas com `3000 <= area <= 150000` e `elongacao <= 4.0`.
Vermelho = rejeitado, amarelo = mantido (anotado `area | elong`). **Unica FALHA real:** uma caixa
**vermelha** que continha uma etiqueta real.

Atencao: Ciano = caixa mantida quase-quadrada. A elongacao e a razao de aspecto do *bounding box*,
**nao** invariante a rotacao (ADR 0004, limitacao conhecida) — um codigo de barras a ~45 graus
parece quadrado e passa. O ciano deixa isso visivel em vez de silencioso.

In [ ]:
NEAR_SQUARE = 1.3
fig, axes = plt.subplots(1, len(PAIR), figsize=(15, 6))
for ax, (label, path, frame) in zip(np.atleast_1d(axes), PAIR):
    working = pdiseg.preprocess(frame)
    candidates = stage1(working)
    kept = pdiseg.keep_label_clusters(candidates)
    kept_set = set(kept)
    rejected = [b for b in candidates if b not in kept_set]
    show(ax, working, f'{label}\nvermelho=rejeitado({len(rejected)})  amarelo=mantido({len(kept)})')

    def ann(boxes):
        return [f'{w * h} | {max(w, h) / min(w, h):.1f}' for _, _, w, h in boxes]

    draw_boxes(ax, rejected, 'red', ann(rejected))
    draw_boxes(ax, kept, 'yellow', ann(kept))
    flagged = [b for b in kept if max(b[2], b[3]) / min(b[2], b[3]) < NEAR_SQUARE]
    draw_boxes(ax, flagged, 'cyan', lw=2.5)
    print(f'{label}: {len(flagged)} caixa(s) mantida(s) quase-quadrada(s)')
plt.tight_layout(); plt.show()
print('FALHA so se uma caixa VERMELHA enquadrava uma etiqueta de nome real.')

## Etapa 2 — `refine_to_name_label`  (split por Otsu, com fallback)

**Objetivo:** encolher cada cluster mantido ate a etiqueta de nome escura.
**Heuristica (ADR 0001):** dentro do cluster so ha duas regioes (badge claro da marca vs. etiqueta
escura), entao o Otsu as separa; pega-se o maior componente escuro. Se o split for ambiguo
(sem regiao escura, ou fracao fora de [0.05, 0.9]), faz **fallback** para o cluster inteiro — que
ainda contem o nome, logo e segmentacao valida e nao falso positivo (rede de seguranca da ADR 0001).

Por cluster: crop, `otsu_dark_mask`, `largest_component_box` escolhido (magenta), caixa refinada
(verde). **BOM** = verde justo no badge; **OK** = fallback ao cluster; **FALHA** = verde numa sombra
ou vao (nome perdido). Imprime a taxa de fallback por regime.

In [ ]:
for label, path, frame in PAIR:
    working = pdiseg.preprocess(frame)
    kept = pdiseg.keep_label_clusters(stage1(working))
    if not kept:
        print(f'{label}: nenhum cluster mantido.'); continue
    n = len(kept)
    fig, axes = plt.subplots(n, 3, figsize=(9, 2.6 * n), squeeze=False)
    fallbacks = 0
    for i, cluster in enumerate(kept):
        x, y, w, h = cluster
        region = working[y:y + h, x:x + w]
        mask = pdiseg.otsu_dark_mask(region)
        local = pdiseg.largest_component_box(mask) if mask is not None else None
        refined = pdiseg.refine_to_name_label(working, cluster)
        is_fb = refined == cluster
        fallbacks += int(is_fb)
        show(axes[i][0], region, f'cluster {i}')
        show(axes[i][1], mask if mask is not None else np.zeros_like(region, bool), 'mascara escura Otsu')
        if local is not None:
            draw_boxes(axes[i][1], [local], 'magenta')
        show(axes[i][2], region, 'OK (fallback)' if is_fb else 'refinado')
        rx, ry, rw, rh = refined
        draw_boxes(axes[i][2], [(rx - x, ry - y, rw, rh)], 'lime', lw=2.2)
    fig.suptitle(f'{label}   fallback = {fallbacks}/{n} = {fallbacks / n:.0%}', fontsize=11)
    plt.tight_layout(); plt.show()

## Placar de ponta a ponta — pipeline PROPOSTO (Stage 1 escura)

**Objetivo:** compor o pipeline proposto (`stage1 -> keep -> refine`) e recortar do frame **original**
(ADR 0003: detecta no preprocessado, recorta do original para o entregavel ficar fiel a fonte —
pixels equalizados/mascarados nunca chegam ao crop). Avalie os crops a olho e preencha as duas
contagens por regime na celula seguinte.

In [ ]:
EMIT = {}
for label, path, frame in PAIR:
    working = pdiseg.preprocess(frame)
    kept = pdiseg.keep_label_clusters(stage1(working))
    boxes = [pdiseg.refine_to_name_label(working, c) for c in kept]
    crops = [pdiseg.crop(frame, b) for b in boxes]
    EMIT[label] = crops
    cols = max(len(crops) + 1, 2)
    fig = plt.figure(figsize=(15, 4))
    ax0 = fig.add_subplot(1, cols, 1)
    show(ax0, working, f'{label}\nverde = emitido ({len(boxes)})')
    draw_boxes(ax0, boxes, 'lime', [str(i + 1) for i in range(len(boxes))], lw=2.0)
    for j, c in enumerate(crops):
        show(fig.add_subplot(1, cols, j + 2), c, f'crop {j + 1}')
    plt.tight_layout(); plt.show()
print('Cada crop contem um nome de produto? Conte crops SEM nome como falsos positivos.')

In [ ]:
# EDITE as duas contagens por regime apos olhar os crops, depois rode.
SCORE = {
    PAIR[0][0]: {'real_labels': 0, 'false_pos': 0},  # FACIL
    PAIR[1][0]: {'real_labels': 0, 'false_pos': 0},  # DIFICIL
}
print(f"{'regime':40} {'emitido':>8} {'real':>6} {'FP':>4} {'recall':>8}")
for label, _, _ in PAIR:
    emitted = len(EMIT[label])
    real = SCORE[label]['real_labels']
    fp = SCORE[label]['false_pos']
    recall = max(emitted - fp, 0) / real if real else float('nan')
    print(f'{label[:40]:40} {emitted:>8} {real:>6} {fp:>4} {recall:>8.0%}')
print('\nrecall = (emitido - falsos positivos) / etiquetas reais, por regime.')

---
# Triagem agregada - rode sobre um escopo, depois filtre

Modo agregado (complementa o mergulho em 2 frames acima). Roda o pipeline proposto sobre um
escopo e classifica cada frame num **bucket de qualidade** (sem ground truth):

- **MERGED** - um candidato cobre >40% do frame (falha de fusao).
- **MISSED** - 0 clusters mantidos (provavel etiqueta perdida).
- **OVERDETECTED** - mais que `OVERDETECT` clusters mantidos.
- **FALLBACK_HEAVY** - a Etapa 2 nao refinou nenhum (todos fallback).
- **OK** - nenhum dos acima.

Depois: tabela-funil **por classe de produto** + celula de **filtro -> grade de overlays**.

## Rodar o pipeline sobre o escopo

In [ ]:
import time
from collections import Counter

# Objetivo: rodar o pipeline proposto sobre um escopo e classificar cada frame num
# bucket de qualidade (sem ground truth, ADR 0004). A heuristica de cada bucket esta em _bucket.
SCOPE = HARD_CLASS         # uma classe, ou 'ALL' (ALL ~3 min / 900 frames)
SCAN_PERCENTILE = STAGE1_PERCENTILE
MERGE_FRAC = 0.40          # candidato cobrindo > isto do frame = fusao (Stage 1 falhou)
OVERDETECT = 10            # mais kept que isto = superdeteccao


def _inspect(working):
    cand = pdiseg.detect_dark_badges(working, percentile=SCAN_PERCENTILE)
    kept = pdiseg.keep_label_clusters(cand)
    labels = [pdiseg.refine_to_name_label(working, c) for c in kept]
    return cand, kept, labels


def _bucket(r):
    if r['biggest_cand_frac'] > MERGE_FRAC:
        return 'MERGED'           # um candidato engole o frame (fusao)
    if r['n_kept'] == 0:
        return 'MISSED'           # nenhum cluster mantido (provavel etiqueta perdida)
    if r['n_kept'] > OVERDETECT:
        return 'OVERDETECTED'     # mantidos demais
    if r['n_labels'] and r['n_fallback'] == r['n_labels']:
        return 'FALLBACK_HEAVY'   # Etapa 2 nao refinou nenhum (tudo fallback)
    return 'OK'


scope_classes = classes() if SCOPE == 'ALL' else [SCOPE]
RESULTS = []
t0 = time.time()
for cls in scope_classes:
    for path in frames(cls):
        frame = load_gray(path)
        h, w = frame.shape
        working = pdiseg.preprocess(frame)
        cand, kept, labels = _inspect(working)
        biggest = max((bw * bh for (_, _, bw, bh) in cand), default=0)
        n_fb = sum(1 for k, l in zip(kept, labels) if l == k)
        rec = {
            'class': cls, 'stem': path.stem, 'path': str(path),
            'n_candidates': len(cand), 'n_kept': len(kept),
            'n_rejected': len(cand) - len(kept), 'n_labels': len(labels),
            'n_fallback': n_fb, 'biggest_cand_frac': biggest / (h * w),
            'cand': cand, 'kept': kept, 'labels': labels,
        }
        rec['bucket'] = _bucket(rec)
        RESULTS.append(rec)
print(f'varridos {len(RESULTS)} frames em {len(scope_classes)} classe(s) '
      f'em {time.time() - t0:.0f}s  (percentil={SCAN_PERCENTILE})')
print('distribuicao de buckets:', dict(Counter(r['bucket'] for r in RESULTS)))

## Tabela-funil por classe de produto

In [ ]:
from collections import Counter, defaultdict

agg = defaultdict(lambda: {'frames': 0, 'cand': 0, 'kept': 0, 'lbl': 0, 'fb': 0,
                           'buckets': Counter()})
for r in RESULTS:
    a = agg[r['class']]
    a['frames'] += 1
    a['cand'] += r['n_candidates']
    a['kept'] += r['n_kept']
    a['lbl'] += r['n_labels']
    a['fb'] += r['n_fallback']
    a['buckets'][r['bucket']] += 1

print(f"{'classe':46}{'frm':>4}{'cand':>6}{'kept':>6}{'lbl':>5}{'fb%':>5}  buckets problematicos")
for cls in sorted(agg):
    a = agg[cls]
    fbp = (a['fb'] / a['lbl'] * 100) if a['lbl'] else 0
    probs = {k: v for k, v in a['buckets'].items() if k != 'OK'}
    print(f"{cls[:46]:46}{a['frames']:>4}{a['cand']:>6}{a['kept']:>6}"
          f"{a['lbl']:>5}{fbp:>4.0f}%  {probs or ''}")

## Filtrar -> grade de overlays

Escolha `FILTER_CLASS` e/ou `FILTER_BUCKET` (`None` = qualquer). Mostra ate `MAX_TILES` overlays dos frames que casam: **vermelho** = candidato rejeitado, **amarelo** = kept, **verde** = name label refinada.

In [ ]:
FILTER_CLASS = None          # uma classe, ou None (qualquer)
FILTER_BUCKET = 'MERGED'     # MISSED|MERGED|OVERDETECTED|FALLBACK_HEAVY|OK|None
MAX_TILES = 12

sel = [r for r in RESULTS
       if FILTER_CLASS in (None, r['class'])
       and FILTER_BUCKET in (None, r['bucket'])]
print(f'{len(sel)} frame(s) casam  classe={FILTER_CLASS}  bucket={FILTER_BUCKET}; '
      f'mostrando ate {MAX_TILES}')
sel = sel[:MAX_TILES]
if sel:
    cols = 3
    rows = (len(sel) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 3.2 * rows), squeeze=False)
    for ax in axes.ravel():
        ax.axis('off')
    for ax, r in zip(axes.ravel(), sel):
        frame = load_gray(r['path'])
        insp = pdiseg.FrameInspection(candidates=r['cand'], kept=r['kept'], labels=r['labels'])
        ax.imshow(pdiseg.render_overlay(frame, insp))
        ax.axis('off')
        ax.set_title(f"{r['class'][:18]}/{r['stem'][-12:]}\n"
                     f"{r['bucket']}  k={r['n_kept']} fb={r['n_fallback']}", fontsize=8)
    plt.tight_layout(); plt.show()

## Depuracao step-by-step por classe (grade filtrada por etapa)

Reusa o `RESULTS` varrido acima (re-rode a celula de scan para trocar `SCOPE`/percentil). Escolha `STEP` para ver QUALQUER etapa do pipeline na grade, filtrada por classe e/ou bucket - assim voce depura uma etapa de cada vez, classe por classe.

`STEP` in: `preprocess | dark_mask | open | close | components | candidates | kept | overlay`

In [ ]:
STEP = 'dark_mask'           # qual etapa renderizar (ver lista acima)
FILTER_CLASS2 = None         # uma classe, ou None (qualquer)
FILTER_BUCKET2 = None        # MISSED|MERGED|OVERDETECTED|FALLBACK_HEAVY|OK|None
MAX_TILES2 = 12


def _step_raster(working, step):
    dm = pdiseg.dark_mask(working, percentile=SCAN_PERCENTILE)
    op = pdiseg.open_mask(dm)
    cl = pdiseg.close_mask(op, structure=np.ones((9, 9)))
    return {'dark_mask': dm, 'open': op, 'close': cl}[step]


def render_step(ax, rec, step):
    frame = load_gray(rec['path'])
    working = pdiseg.preprocess(frame)
    if step == 'preprocess':
        ax.imshow(working, cmap='gray', vmin=0, vmax=255)
    elif step in ('dark_mask', 'open', 'close'):
        ax.imshow(_step_raster(working, step), cmap='gray')
    elif step == 'components':
        comps = pdiseg.label_components(_step_raster(working, 'close'))
        ax.imshow(np.where(comps > 0, comps, np.nan), cmap='nipy_spectral')
    elif step == 'candidates':
        ax.imshow(working, cmap='gray', vmin=0, vmax=255)
        draw_boxes(ax, rec['cand'], 'yellow', lw=1.0)
    elif step == 'kept':
        ax.imshow(working, cmap='gray', vmin=0, vmax=255)
        ks = set(rec['kept'])
        draw_boxes(ax, [b for b in rec['cand'] if b not in ks], 'red', lw=1.0)
        draw_boxes(ax, rec['kept'], 'yellow', lw=1.8)
    elif step == 'overlay':
        insp = pdiseg.FrameInspection(candidates=rec['cand'], kept=rec['kept'], labels=rec['labels'])
        ax.imshow(pdiseg.render_overlay(frame, insp))
    else:
        raise ValueError(f'etapa desconhecida: STEP={step!r}')
    ax.axis('off')


sel = [r for r in RESULTS
       if FILTER_CLASS2 in (None, r['class'])
       and FILTER_BUCKET2 in (None, r['bucket'])]
print(f"STEP={STEP}  casam={len(sel)}  classe={FILTER_CLASS2}  bucket={FILTER_BUCKET2}; "
      f'mostrando ate {MAX_TILES2}')
sel = sel[:MAX_TILES2]
if sel:
    cols = 3
    rows = (len(sel) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 3.2 * rows), squeeze=False)
    for ax in axes.ravel():
        ax.axis('off')
    for ax, r in zip(axes.ravel(), sel):
        render_step(ax, r, STEP)
        ax.set_title(f"{r['class'][:16]}/{r['stem'][-10:]}\n{r['bucket']}  k={r['n_kept']}", fontsize=8)
    plt.tight_layout(); plt.show()